# M06 · os, NumPy & pandas for Bioinformatics (Comprehensive Assignment)
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M06_os_NumPy_pandas/M06_Assignment.ipynb)

⏱ ~7 hours · Difficulty: ★★☆ · Prerequisites: M01 (Basics of Python), M02 (File Handling), M04 (Visualisation)

---

## 1. Learning Outcomes

By completing this comprehensive assignment, you will be able to:
1. **Automate filesystem workflows** (`os`, `pathlib`, `shutil`) to traverse nested genomic directories, catalog file sizes, count sequence records, and batch-organize raw sequencing outputs.
2. **Apply regular expressions to file automation** to batch-rename mis-sorted sample files into standardized, zero-padded identifier conventions.
3. **Vectorize genomic algorithms in NumPy** to replace slow Python loops, achieving 50–100× speedups on sliding-window GC profiles and k-mer spectra.
4. **Construct biological tensor representations** including one-hot encoded DNA matrices, Position Frequency Matrices (PFMs), and Position Weight Matrices (PWMs) with information content in bits.
5. **Normalize high-throughput expression matrices** using array broadcasting for $\log_2$ transformations and gene-wise $Z$-score standardization across samples.
6. **Wrangle, filter, merge, and tidy multidimensional genomics tables** using pandas DataFrames and construct publication-ready multi-condition comparative plots with Seaborn.
7. **Perform advanced bioinformatics tabular operations** including MultiIndex aggregations across tissue cohorts, differential expression volcano plot filtering, and gene pathway enrichment grouping.

---

## 2. Prerequisites

Ensure you are familiar with:
- **Module 1 (Python Basics):** Slicing, collections, functions, list/dict comprehensions.
- **Module 2 (File Handling & Biopython):** Reading files, context managers, parsing FASTA with `Bio.SeqIO`.
- **Module 4 (Data Visualisation):** Matplotlib figures, Seaborn syntax, color palettes.


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os, shutil, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import Bio
from Bio import SeqIO
import matplotlib.pyplot as plt
import seaborn as sns

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    current = Path.cwd().resolve()
    ROOT = None
    for p in [current, *current.parents]:
        if (p / "COURSE_PLAN.md").exists() or (p / ".git").exists():
            ROOT = p
            break
    if ROOT is None:
        ROOT = current

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M06"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Global plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print(f"Python:    {sys.version.split()[0]}")
print(f"NumPy:     {np.__version__}")
print(f"pandas:    {pd.__version__}")
print(f"Biopython: {Bio.__version__}")
print(f"ROOT:      {ROOT.resolve()}")
print(f"DATA_DIR:  {DATA_DIR.resolve()} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:   {OUT_DIR.resolve()}")


---

# Part A · File System Automation (`os`, `pathlib`, `shutil`)

### 4.A.1 Theory: The File Management Challenge in Computational Biology
Modern genomics workflows generate massive volumes of distributed files: Illumina raw FASTQ files, BAM alignments, GFF3 gene annotations, VCF variant calls, and FASTA references.

Computational biologists must programmatically:
- Discover and catalog files across nested directory trees without hardcoded paths.
- Parse file sizes, extensions, and metadata to verify download completeness.
- Safely organize files into categorized directory structures without accidental deletion.
- Batch-rename files to comply with standardized sample accession numbers.

| Operation | Legacy `os` / `os.path` | Modern `pathlib.Path` (`p = Path(...)`) | Notes |
|---|---|---|---|
| Current working directory | `os.getcwd()` | `Path.cwd()` | Returns current directory |
| Join paths | `os.path.join(d, "file.fa")` | `d / "file.fa"` | Forward-slash operator handles Windows/POSIX |
| Check existence | `os.path.exists(path)` | `p.exists()` | Returns boolean |
| File size in bytes | `os.path.getsize(path)` | `p.stat().st_size` | Integer bytes |
| Filename and extension | `os.path.splitext("a.fa")` | `p.stem`, `p.suffix` | `.stem` -> `"a"`, `.suffix` -> `".fa"` |
| Create directory | `os.makedirs(d, exist_ok=True)` | `p.mkdir(parents=True, exist_ok=True)` | Safe recursive directory creation |
| Recursive search | `os.walk(d)` | `p.rglob("*.fasta")` | Generator yielding matching paths |
| Copy file | `shutil.copy2(src, dst)` | `shutil.copy2(src, dst)` | Copies data and metadata (timestamps) |
| Move / rename | `shutil.move(src, dst)` | `p.rename(new_path)` | Moves across or renames files |


In [ ]:
# 4.A.2 Worked Example: Inspecting and traversing directory paths with pathlib
example_path = DATA_DIR / "sequences" / "multi_sample.fasta"

print("--- Path Anatomy ---")
print(f"Full path:       {example_path}")
print(f"Parent folder:   {example_path.parent}")
print(f"File name:       {example_path.name}")
print(f"Stem (no ext):   {example_path.stem}")
print(f"Suffix:          {example_path.suffix}")
print(f"Exists on disk?  {example_path.exists()}")
print(f"File size:       {example_path.stat().st_size:,} bytes")

print("\n--- Directory Discovery with rglob ---")
csv_files = list(DATA_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV tables under {DATA_DIR.name}/:")
for csv in csv_files[:4]:
    print(f"  • {csv.relative_to(ROOT)} ({csv.stat().st_size} bytes)")


In [ ]:
# 4.A.3 ✏️ Try It: Count specific file formats
def count_files_by_extension(directory: Path, extension: str) -> int:
    ext = extension.lower() if extension.startswith(".") else f".{extension.lower()}"
    return sum(1 for p in Path(directory).iterdir() if p.is_file() and p.suffix.lower() == ext)

num_csv = count_files_by_extension(DATA_DIR / "tables", ".csv")
print(f"CSV files in data/tables: {num_csv}")
assert num_csv >= 4, f"Expected at least 4 CSV files, found {num_csv}"
print("Try It passed ✔")


### 4.A.4 ⚠️ Common Mistakes in File & Path Handling
> [!WARNING]
> 1. **Hardcoding slashes (`/` or `\\`):** Always use `pathlib.Path` with division operator `folder / filename`.
> 2. **Modifying a directory while iterating over it:** Always freeze paths into a list: `files = list(p.iterdir())` before deleting or renaming.
> 3. **Assuming `Path.cwd()` is repo root:** Always find repo root dynamically.
> 4. **Silently Overwriting Data:** Check `if not dst.exists():` before overwriting.


---

## Graded Task A1 (★☆☆) · Walk Data Directory & Catalog FASTA Records

### Specification
Implement `walk_fasta_records(data_dir: Path) -> pd.DataFrame`:
1. Recursively search `data_dir` for files with extensions `[".fasta", ".fa", ".fna"]`.
2. Extract `filename`, `relative_path` (relative to `data_dir`), `size_bytes`, and `num_records` (via `SeqIO.parse`).
3. Return a DataFrame sorted descending by `size_bytes`.


In [ ]:
def walk_fasta_records(data_dir: Path) -> pd.DataFrame:
    # TODO: Walk data_dir recursively for FASTA files, count records with SeqIO,
    # and return a DataFrame sorted descending by size_bytes.
    ...


In [ ]:
# ✅ Test Task A1 — Run me to verify
df_fasta = walk_fasta_records(DATA_DIR)
print(df_fasta)

assert isinstance(df_fasta, pd.DataFrame)
assert set(["filename", "relative_path", "size_bytes", "num_records"]).issubset(df_fasta.columns)
assert len(df_fasta) >= 2
assert "multi_sample.fasta" in df_fasta["filename"].values
assert "tp53_human_mrna.fasta" in df_fasta["filename"].values
assert df_fasta[df_fasta["filename"] == "multi_sample.fasta"].iloc[0]["num_records"] == 15
assert df_fasta[df_fasta["filename"] == "tp53_human_mrna.fasta"].iloc[0]["num_records"] == 1
assert df_fasta["size_bytes"].is_monotonic_decreasing
print("Task A1 passed ✔")


---

## Graded Task A2 (★★☆) · Organize a Messy Staging Directory

### Specification
Implement `organize_directory(source_dir: Path, target_dir: Path) -> dict[str, int]`:
1. Categorize files in `source_dir` into `target_dir`:
   - `fasta/`: `.fasta`, `.fa`, `.fna`
   - `genbank/`: `.gb`, `.gp`, `.gbk`
   - `tables/`: `.csv`, `.tsv`, `.txt`
   - `other/`: all other extensions
2. Copy files using `shutil.copy2` to preserve modification timestamps.
3. Return a dictionary of counts: `{"fasta": count, "genbank": count, "tables": count, "other": count}`.


In [ ]:
def organize_directory(source_dir: Path, target_dir: Path) -> dict[str, int]:
    # TODO: Categorize files in source_dir into target_dir subfolders.
    ...


In [ ]:
# ✅ Test Task A2 — Run me to verify
staging_src = OUT_DIR / "test_staging_messy"
staging_tgt = OUT_DIR / "test_staging_organized"
shutil.rmtree(staging_src, ignore_errors=True)
shutil.rmtree(staging_tgt, ignore_errors=True)
staging_src.mkdir(parents=True, exist_ok=True)

(staging_src / "sample_A.fasta").write_text(">A\nATGC")
(staging_src / "sample_B.fa").write_text(">B\nGGCC")
(staging_src / "locus_1.gb").write_text("LOCUS dummy")
(staging_src / "counts.csv").write_text("gene,count\nTP53,100")
(staging_src / "metadata.tsv").write_text("id\tbatch\n1\tA")
(staging_src / "readme.txt").write_text("Project notes")
(staging_src / "config.yaml").write_text("threads: 4")

summary = organize_directory(staging_src, staging_tgt)
assert summary["fasta"] == 2
assert summary["genbank"] == 1
assert summary["tables"] == 3
assert summary["other"] == 1
assert (staging_tgt / "fasta" / "sample_A.fasta").exists()
assert (staging_tgt / "genbank" / "locus_1.gb").exists()
print("Task A2 passed ✔")


---

## Graded Task A3 (★★☆) · Batch-Rename Sequencing Samples with Zero-Padding

### Specification
Implement `batch_rename_samples(directory: Path, pattern: str = r"^sample(\d+)\.(fa|fasta)$") -> list[tuple[str, str]]`:
1. Find files matching regex `pattern`.
2. Rename each to `f"S{index:03d}.fasta"`.
3. Return sorted list of `(old_name, new_name)` tuples.


In [ ]:
def batch_rename_samples(directory: Path, pattern: str = r"^sample(\d+)\.(fa|fasta)$") -> list[tuple[str, str]]:
    # TODO: Batch-rename files matching pattern to S{idx:03d}.fasta
    ...


In [ ]:
# ✅ Test Task A3 — Run me to verify
rename_dir = OUT_DIR / "test_batch_rename"
shutil.rmtree(rename_dir, ignore_errors=True)
rename_dir.mkdir(parents=True, exist_ok=True)

(rename_dir / "sample1.fa").write_text(">S1\nATGC")
(rename_dir / "sample2.fa").write_text(">S2\nCGTA")
(rename_dir / "sample10.fa").write_text(">S10\nTTTT")
(rename_dir / "sample105.fasta").write_text(">S105\nGGGG")

renamed_pairs = batch_rename_samples(rename_dir)
assert len(renamed_pairs) == 4
assert ("sample1.fa", "S001.fasta") in renamed_pairs
assert ("sample2.fa", "S002.fasta") in renamed_pairs
assert (rename_dir / "S001.fasta").exists()
assert not (rename_dir / "sample1.fa").exists()
print("Task A3 passed ✔")


---

# Part B · High-Performance Numerical Computing with NumPy

### 4.B.1 Theory: Vectorization vs Python Loops in Bioinformatics
Standard Python lists wrap each number in a 28-byte `PyObject` scattered across memory. **NumPy** stores data in contiguous C buffers with homogeneous `dtype`, executing SIMD instructions in hardware.

| Concept | Python Loop | NumPy Vectorized | Speedup |
|---|---|---|---|
| Sliding Window GC | Slicing in `for` loop | `np.cumsum` prefix sums | 50× – 100× |
| $\log_2$ Transformation | `[math.log2(x + 1) for x in counts]` | `np.log2(counts + 1)` | 20× – 50× |
| Z-score Normalization | Nested loops for mean/std | Array broadcasting `(mat - mu) / sigma` | 40× – 80× |


In [ ]:
# 4.B.2 Worked Example: Array Creation, Broadcasting, and Axis Reductions
raw_counts = np.array([
    [120.0, 150.0, 110.0, 140.0],
    [  5.0,   2.0,   0.0,   1.0],
    [ 45.0,  95.0,  40.0,  90.0]
], dtype=np.float64)

log2_expr = np.log2(raw_counts + 1.0)
gene_means = np.mean(log2_expr, axis=1, keepdims=True)
mean_centered = log2_expr - gene_means
print("Original shape:", raw_counts.shape)
print("Gene means shape:", gene_means.shape)
print("Mean centered:\n", np.round(mean_centered, 2))


In [ ]:
# 4.B.3 ✏️ Try It: Vectorized Purine Fraction
def purine_fraction_vectorized(seq: str) -> float:
    if not seq:
        return 0.0
    arr = np.frombuffer(seq.encode("ascii"), dtype=np.uint8)
    is_purine = (arr == ord('A')) | (arr == ord('a')) | (arr == ord('G')) | (arr == ord('g'))
    return float(np.mean(is_purine))

pf = purine_fraction_vectorized("AGTCAGTC")
assert pf == 0.5
print("Try It passed ✔")


### 4.B.4 ⚠️ Common Mistakes in NumPy
> [!WARNING]
> 1. **Modifying Views Accidentally:** Slicing produces a view. Use `.copy()` when you need independence.
> 2. **Forgetting `keepdims=True`:** Preserves dimensions for clean row/column broadcasting.
> 3. **Using `math` functions on arrays:** Use universal functions (`np.log2`, `np.exp`).


---

## Graded Task B1 (★★☆) · One-Hot Encoding DNA Sequences

### Specification
Implement `one_hot_encode_dna(seq: str) -> np.ndarray`:
- Encode DNA into $(L \times 4)$ binary array: $A=[1,0,0,0], C=[0,1,0,0], G=[0,0,1,0], T=[0,0,0,1]$.
- Non-ACGT bases $\to [0,0,0,0]$.
- Return `np.ndarray` of dtype `np.int8`.


In [ ]:
def one_hot_encode_dna(seq: str) -> np.ndarray:
    # TODO: One-hot encode DNA into (L, 4) binary array
    ...


In [ ]:
# ✅ Test Task B1 — Run me to verify
enc = one_hot_encode_dna("ACGTacgtN-")
assert enc.shape == (10, 4)
assert enc.dtype == np.int8
assert np.array_equal(enc[0], [1, 0, 0, 0])
assert np.array_equal(enc[1], [0, 1, 0, 0])
assert np.array_equal(enc[8], [0, 0, 0, 0])
assert one_hot_encode_dna("").shape == (0, 4)
print("Task B1 passed ✔")


---

## Graded Task B2 (★★★) · Vectorized Sliding-Window GC Content & Benchmarking

### Specification
Implement `sliding_window_gc_vectorized(seq: str, window_size: int = 100, step: int = 1) -> tuple[np.ndarray, np.ndarray]`:
- Calculate sliding window GC using prefix sums (`np.cumsum`) in $O(N)$ time.
- Return `(starts, gc_percentages)`.


In [ ]:
def sliding_window_gc_vectorized(seq: str, window_size: int = 100, step: int = 1) -> tuple[np.ndarray, np.ndarray]:
    # TODO: Implement O(N) sliding window GC using np.cumsum.
    ...


In [ ]:
# ✅ Test Task B2 & Benchmark
test_seq = "ATGC" * 5000  # 20,000 bp
starts, gc_v = sliding_window_gc_vectorized(test_seq, window_size=100, step=10)
assert len(starts) == (len(test_seq) - 100) // 10 + 1
assert np.allclose(gc_v, 50.0)

def loop_gc(s, w=100, step=10):
    res = []
    for i in range(0, len(s) - w + 1, step):
        sub = s[i:i+w].upper()
        res.append((sub.count('G') + sub.count('C')) / w * 100.0)
    return np.array(res)

t0 = time.perf_counter()
loop_gc(test_seq)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
sliding_window_gc_vectorized(test_seq, 100, 10)
t_vec = time.perf_counter() - t0

speedup = t_loop / t_vec
print(f"Loop: {t_loop*1000:.2f} ms | Vectorized: {t_vec*1000:.2f} ms | Speedup: {speedup:.1f}x")
assert speedup > 2.0  # Vectorized implementation is demonstrably faster
print("Task B2 passed ✔")


---

## Graded Task B3 (★★☆) · Expression Normalization: Log2 & Gene-Wise Z-Score

### Specification
Implement `normalize_expression(matrix: np.ndarray, pseudocount: float = 1.0) -> tuple[np.ndarray, np.ndarray]`:
- Compute $Y = \log_2(\text{matrix} + \text{pseudocount})$.
- Compute gene-wise $Z$-scores across samples along `axis=1`.
- Handle zero-variance genes safely (set $Z = 0.0$).
- Return `(log2_matrix, zscore_matrix)`.


In [ ]:
def normalize_expression(matrix: np.ndarray, pseudocount: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    # TODO: Log2 transform and gene-wise z-score across axis=1.
    ...


In [ ]:
# ✅ Test Task B3 — Run me to verify
sample_mat = np.array([
    [10.0, 20.0, 30.0, 40.0],
    [100.0, 100.0, 100.0, 100.0],
    [5.0, 15.0, 25.0, 35.0]
])
l2_m, z_m = normalize_expression(sample_mat)
assert l2_m.shape == (3, 4)
assert z_m.shape == (3, 4)
assert np.allclose(z_m[1], 0.0)
assert np.allclose(np.mean(z_m[0]), 0.0, atol=1e-7)
assert np.allclose(np.std(z_m[0]), 1.0, atol=1e-7)
print("Task B3 passed ✔")


---

## Graded Task B4 (★★☆) · Build a Position Frequency Matrix (PFM)

### Specification
Implement `build_position_frequency_matrix(aligned_sequences: list[str]) -> np.ndarray`:
- Given $N$ aligned DNA strings of length $L$, compute $(4 \times L)$ frequency matrix.
- Rows: $0 \to A, 1 \to C, 2 \to G, 3 \to T$. Columns sum to $1.0$.


In [ ]:
def build_position_frequency_matrix(aligned_sequences: list[str]) -> np.ndarray:
    # TODO: Build (4, L) PFM from aligned sequences.
    ...


In [ ]:
# ✅ Test Task B4 — Run me to verify
tata = ["TATAAA", "TATATA", "TACAAA", "TATAAA"]
pfm = build_position_frequency_matrix(tata)
assert pfm.shape == (4, 6)
assert np.allclose(pfm.sum(axis=0), 1.0)
assert pfm[3, 0] == 1.0
assert pfm[0, 1] == 1.0
print("Task B4 passed ✔")


---

## Graded Task B5 (★★★) · Position Weight Matrix (PWM) & Information Content

### Biological Context
A Position Weight Matrix (PWM) transforms raw frequencies into **log-odds scores** relative to a genomic background nucleotide distribution $B = [B_A, B_C, B_G, B_T]$:
$$W_{i, j} = \log_2\left(\frac{P_{i, j}}{B_i}\right)$$
To avoid $\log_2(0)$, we add a small pseudocount $\epsilon$ (e.g. $0.01$).

The **Information Content (IC)** in bits at position $j$ measures sequence conservation:
$$IC(j) = 2 - H(j) = 2 - \left(-\sum_{i=1}^{4} P_{i, j} \log_2 P_{i, j}\right)$$
Positions with $IC \approx 2$ bits are strictly conserved; positions with $IC \approx 0$ bits exhibit random nucleotide distribution.

### Specification
Implement `build_position_weight_matrix(pfm: np.ndarray, background: np.ndarray = None, pseudocount: float = 0.01) -> tuple[np.ndarray, np.ndarray]`:
1. Default background: $B = [0.25, 0.25, 0.25, 0.25]$.
2. Regularize probabilities: $P = \frac{\text{pfm} + \epsilon}{\sum (\text{pfm} + \epsilon)}$.
3. Compute log-odds weights: $W = \log_2(P / B)$.
4. Compute Shannon entropy $H = -\sum P \log_2(P)$ and $IC = 2 - H$.
5. Return `(pwm, ic_vector)`.


In [ ]:
def build_position_weight_matrix(
    pfm: np.ndarray,
    background: np.ndarray = None,
    pseudocount: float = 0.01
) -> tuple[np.ndarray, np.ndarray]:
    # TODO: Compute log-odds PWM and Information Content vector in bits.
    ...


In [ ]:
# ✅ Test Task B5 — Run me to verify
pfm_test = np.array([
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
    [1.0, 0.0, 1.0, 1.0]
])
pwm, ic = build_position_weight_matrix(pfm_test)
print("PWM shape:", pwm.shape)
print("Information Content (bits):", np.round(ic, 2))

assert pwm.shape == (4, 4)
assert len(ic) == 4
assert np.all(ic > 1.7), "Conserved positions should have IC > 1.7 bits"
print("Task B5 passed ✔")


---

## Graded Task B6 (★★★) · Vectorized k-mer Counting with NumPy Strides

### Biological Context
k-mer spectra ($k=3, 5, 21, 31$) form the computational backbone of genome assemblers (de Bruijn graphs), metagenomic taxonomic classifiers (Kraken), and alignment-free distance estimators (Mash). Extracting overlapping k-mers using naive string slicing takes $O(N \times k)$ in pure Python. Using NumPy's **stride tricks** (`sliding_window_view`), overlapping k-mers can be created with zero-copy memory views.

### Specification
Implement `count_kmers_vectorized(seq: str, k: int = 3) -> dict[str, int]`:
1. If `len(seq) < k`, return an empty dictionary `{}`.
2. Convert `seq` to uppercase ASCII byte array.
3. Use `np.lib.stride_tricks.sliding_window_view(seq_bytes, window_shape=k)` to generate all contiguous length-$k$ subwindows.
4. Decode subwindows back to strings and count frequencies using `np.unique(..., return_counts=True)`.
5. Return a dictionary mapping `{kmer_string: count}`.


In [ ]:
def count_kmers_vectorized(seq: str, k: int = 3) -> dict[str, int]:
    # TODO: Extract k-mers using sliding_window_view and return frequency dictionary.
    ...


In [ ]:
# ✅ Test Task B6 — Run me to verify
k_counts = count_kmers_vectorized("ATGCATGCATGC", k=3)
print("k-mer counts (k=3):", k_counts)

assert k_counts["ATG"] == 3
assert k_counts["TGC"] == 3
assert k_counts["GCA"] == 2
assert k_counts["CAT"] == 2
assert count_kmers_vectorized("AT", k=3) == {}
print("Task B6 passed ✔")


---

## Graded Task B7 (★★☆) · Vectorized Transition/Transversion ($Ti/Tv$) Ratio

### Biological Context
In molecular evolution, nucleotide substitutions are classified into:
- **Transitions ($Ti$):** Exchange between identical ring classes:
  - Purine $\leftrightarrow$ Purine ($A \leftrightarrow G$)
  - Pyrimidine $\leftrightarrow$ Pyrimidine ($C \leftrightarrow T$)
- **Transversions ($Tv$):** Exchange between different ring classes:
  - Purine $\leftrightarrow$ Pyrimidine ($A \leftrightarrow C, A \leftrightarrow T, G \leftrightarrow C, G \leftrightarrow T$)

Although there are twice as many possible transversions as transitions (8 vs 4), transitions occur at 2–4× higher frequencies in natural genomes due to tautomeric shifts and cytosine deamination. The $Ti/Tv$ ratio serves as a critical QC metric for variant calling pipelines (e.g. GATK).

### Specification
Implement `transition_transversion_ratio(seq1: str, seq2: str) -> float`:
1. Verify `len(seq1) == len(seq2)`. If length is $0$, return $0.0$.
2. Convert both sequences to ASCII byte arrays.
3. Vectorially identify mismatch positions between canonical bases ($A, C, G, T$).
4. Compute transition mask ($Ti$) and transversion mask ($Tv$).
5. Return $\frac{Ti}{Tv}$ as a float. If $Tv = 0$, return `float('inf')` if $Ti > 0$ else $0.0$.


In [ ]:
def transition_transversion_ratio(seq1: str, seq2: str) -> float:
    # TODO: Vectorized calculation of Ti/Tv ratio between two aligned DNA strings.
    ...


In [ ]:
# ✅ Test Task B7 — Run me to verify
s1 = "ACGTACGTACGT"
s2 = "GCATATGTACGA"  # Ti=3, Tv=1 -> 3.0
r = transition_transversion_ratio(s1, s2)
print("Computed Ti/Tv ratio:", r)

assert r == 3.0
assert transition_transversion_ratio("A", "G") == float("inf")
assert transition_transversion_ratio("A", "C") == 0.0
assert transition_transversion_ratio("A", "A") == 0.0
print("Task B7 passed ✔")


---

# Part C · Tabular Data Science with pandas

### 4.C.1 Theory: The Tabular Lingua Franca of Computational Biology
Tabular datasets dominate bioinformatics: RNA-seq count matrices, clinical cohort metadata, GFF3 gene annotations, variant call tables, and differential expression summaries.

The **pandas** library provides:
- `pd.Series`: 1D labeled array.
- `pd.DataFrame`: 2D tabular data structure with heterogeneous typed columns and labeled axes.

| Operation | Syntax | Biological Application |
|---|---|---|
| Read CSV / TSV | `pd.read_csv("file.tsv", sep="\t")` | Load expression tables |
| Label-based slice | `df.loc[genes, ["Control_1", "Treated_1"]]` | Query specific genes by symbol |
| Integer slice | `df.iloc[0:10, 0:5]` | Slice top 10 rows and first 5 columns |
| Boolean filtering | `df[(df["mean"] > 10) & (df["padj"] < 0.05)]` | Select statistically significant genes |
| Aggregation | `df.groupby("chromosome")["tpm"].mean()` | Average expression per chromosome |
| Relational Merge | `pd.merge(expr_df, annot_df, on="gene")` | Join expression with genomic coordinates |
| Reshape (Wide $\to$ Long) | `df.melt(id_vars=["gene"], var_name="sample")` | Prepare data for Seaborn violin/boxplots |


In [ ]:
# 4.C.2 Worked Example: Inspecting, Filtering, and Grouping Tabular Data
example_annot = DATA_DIR / "tables" / "gene_annotation.csv"
df_annot = pd.read_csv(example_annot)
df_annot["gene_length_bp"] = df_annot["end"] - df_annot["start"]

fwd_coding = df_annot[(df_annot["biotype"] == "protein_coding") & (df_annot["strand"] == "+")]
print(f"Forward-strand protein-coding genes: {len(fwd_coding)}")
chr_summary = df_annot.groupby("chromosome")["gene_length_bp"].agg(["count", "mean", "median"])
print(chr_summary.head(4))


In [ ]:
# 4.C.3 ✏️ Try It: Filter genes by chromosome
target_genes = df_annot[
    (df_annot["chromosome"].isin(["chr1", "chr2"])) & 
    (df_annot["gene_length_bp"] > 20000)
]
print(f"Discovered {len(target_genes)} large genes on chr1 and chr2")
assert len(target_genes) >= 1
print("Try It passed ✔")


### 4.C.4 ⚠️ Common Mistakes in pandas
> [!WARNING]
> 1. **`SettingWithCopyWarning`:** Always call `.copy()` when slicing a DataFrame subset.
> 2. **Chained Indexing:** Use `.loc[row, col]` instead of `df['col'][row]`.
> 3. **Type Mismatch on Merge:** Verify column dtypes match before calling `pd.merge`.


---

## Graded Task C1 (★★☆) · Filter Expression Matrix & Identify Highly Variable Genes

### Specification
Implement `analyze_expression_matrix(filepath: Path, mean_threshold: float = 100.0) -> tuple[pd.DataFrame, pd.DataFrame]`:
1. Calculate `mean_expression` and `variance` across sample columns for each gene.
2. Filter genes where `mean_expression > mean_threshold`.
3. Extract top 10 genes by variance sorted descending.
4. Return `(filtered_df, top10_var_df)`.


In [ ]:
def analyze_expression_matrix(filepath: Path, mean_threshold: float = 100.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    # TODO: Load expression matrix, calculate mean & variance, filter and extract top 10 variable genes.
    ...


In [ ]:
# ✅ Test Task C1 — Run me to verify
expr_file = DATA_DIR / "tables" / "expression_matrix.csv"
filt_df, top10_df = analyze_expression_matrix(expr_file, mean_threshold=100.0)

assert isinstance(filt_df, pd.DataFrame)
assert len(top10_df) == 10
assert top10_df["variance"].is_monotonic_decreasing
print("Task C1 passed ✔")


---

## Graded Task C2 (★★☆) · Merge Expression with Genomic Annotations & Chromosome Grouping

### Specification
Implement `merge_expression_with_annotations(expr_df: pd.DataFrame, annot_filepath: Path) -> tuple[pd.DataFrame, pd.DataFrame]`:
1. Merge on `'gene'`.
2. Add `'gene_length'` (`end - start`).
3. Group by `'chromosome'` and calculate `count`, `mean`, `median`, `std` on `'mean_expression'`.
4. Return `(merged_df, chr_summary_df)`.


In [ ]:
def merge_expression_with_annotations(expr_df: pd.DataFrame, annot_filepath: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    # TODO: Merge on gene, add gene_length, compute chromosome summary stats.
    ...


In [ ]:
# ✅ Test Task C2 — Run me to verify
annot_file = DATA_DIR / "tables" / "gene_annotation.csv"
merged_df, chr_summary = merge_expression_with_annotations(filt_df, annot_file)

assert "gene_length" in merged_df.columns
assert "chromosome" in chr_summary.columns
assert set(["count", "mean", "median", "std"]).issubset(chr_summary.columns)
print("Task C2 passed ✔")


---

## Graded Task C3 (★★☆) · FASTA to pandas: Sequence Feature Extraction & CSV Export

### Specification
Implement `fasta_to_dataframe(fasta_filepath: Path, output_csv: Path) -> pd.DataFrame`:
1. Parse FASTA records via `SeqIO.parse`.
2. Extract `id`, `description`, `length`, `gc_percent`, `at_skew`, `purine_fraction`.
3. Save to `output_csv` and return DataFrame.


In [ ]:
def fasta_to_dataframe(fasta_filepath: Path, output_csv: Path) -> pd.DataFrame:
    # TODO: Parse FASTA, compute sequence statistics, save to CSV, return DataFrame.
    ...


In [ ]:
# ✅ Test Task C3 — Run me to verify
multi_fa = DATA_DIR / "sequences" / "multi_sample.fasta"
out_csv = OUT_DIR / "fasta_summary.csv"
df_fasta_summary = fasta_to_dataframe(multi_fa, out_csv)

assert len(df_fasta_summary) == 15
assert out_csv.exists()
assert "purine_fraction" in df_fasta_summary.columns
print("Task C3 passed ✔")


---

## Graded Task C4 (★★★) · Tidy Expression Reshaping & Publication Seaborn Plot

### Specification
Implement `tidy_expression_and_plot(matrix_filepath: Path, top_n_genes: int = 5, save_path: Path = None) -> pd.DataFrame`:
1. Reshape wide expression matrix to tidy format with columns: `['gene', 'sample', 'condition', 'replicate', 'expression']`.
2. Filter for top `top_n_genes` genes by mean expression.
3. If `save_path` is given, plot boxplot with jittered stripplot and save.
4. Return tidy DataFrame.


In [ ]:
def tidy_expression_and_plot(matrix_filepath: Path, top_n_genes: int = 5, save_path: Path = None) -> pd.DataFrame:
    # TODO: Melt matrix, extract condition & replicate, generate Seaborn plot, return tidy DataFrame.
    ...


In [ ]:
# ✅ Test Task C4 — Run me to verify
plot_out = OUT_DIR / "top_genes_tidy.png"
tidy_df = tidy_expression_and_plot(expr_file, top_n_genes=5, save_path=plot_out)

assert isinstance(tidy_df, pd.DataFrame)
assert set(["gene", "sample", "condition", "replicate", "expression"]).issubset(tidy_df.columns)
assert set(tidy_df["condition"].unique()) == {"Control", "Treated"}
assert plot_out.exists()
print("Task C4 passed ✔")


---

## Graded Task C5 (★★☆) · Multi-Index Aggregations & Tissue Comparison Matrix

### Biological Context
Mammalian gene expression is highly tissue-specific. A dataset recording expression across multiple tissues and biological replicates (`data/tables/expression_tissues.csv`) requires hierarchical aggregation:
1. Summarizing replicate variability within each tissue cohort.
2. Pivoting into a clean $(Genes \times Tissues)$ comparison matrix to evaluate tissue-specificity indices.

### Specification
Implement `multi_index_tissue_analysis(filepath: Path) -> tuple[pd.DataFrame, pd.DataFrame]`:
1. Load `filepath` (`expression_tissues.csv`, with columns `gene`, `tissue`, `replicate`, `tpm`).
2. Group by `["tissue", "gene"]` and compute `["mean", "std", "count"]` of `tpm`. Reset the index to produce `grouped_df`.
3. Construct a pivot table `pivot_matrix` with `index="gene"`, `columns="tissue"`, and `values="tpm"` aggregated by `mean`.
4. Return `(grouped_df, pivot_matrix)`.


In [ ]:
def multi_index_tissue_analysis(filepath: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    # TODO: MultiIndex grouping on (tissue, gene) and pivot into Gene x Tissue matrix.
    ...


In [ ]:
# ✅ Test Task C5 — Run me to verify
tissues_csv = DATA_DIR / "tables" / "expression_tissues.csv"
grp_df, pvt_mat = multi_index_tissue_analysis(tissues_csv)
print("Tissue pivot matrix preview:\n", pvt_mat.head(3))

assert isinstance(grp_df, pd.DataFrame)
assert isinstance(pvt_mat, pd.DataFrame)
assert "mean" in grp_df.columns
assert pvt_mat.shape == (5, 5)
assert "Brain" in pvt_mat.columns
print("Task C5 passed ✔")


---

## Graded Task C6 (★★★) · Differential Expression Volcano Filtering & Classification

### Biological Context
A volcano plot plots statistical significance ($-\log_{10}(p_{\text{adj}})$) on the y-axis against biological effect size ($\log_2\text{FC}$) on the x-axis. 
Genes are classified into three distinct regulatory regimes:
- **Up-regulated:** $\log_2\text{FC} \ge \text{lfc\_thresh}$ and $p_{\text{adj}} < \text{padj\_thresh}$
- **Down-regulated:** $\log_2\text{FC} \le -\text{lfc\_thresh}$ and $p_{\text{adj}} < \text{padj\_thresh}$
- **Not-significant:** Fails either effect size or significance threshold

### Specification
Implement `filter_differential_expression(filepath: Path, lfc_thresh: float = 1.0, padj_thresh: float = 0.05, save_path: Path = None) -> tuple[pd.DataFrame, dict[str, int]]`:
1. Load `data/tables/differential_expression.csv`.
2. Add a column `'regulation'` assigning `'Up-regulated'`, `'Down-regulated'`, or `'Not-significant'`.
3. Compute a summary counts dictionary: `{"Up-regulated": n_up, "Down-regulated": n_down, "Not-significant": n_ns}`.
4. If `save_path` is given, plot a volcano plot using Seaborn scatterplot with threshold lines and save at 300 DPI.
5. Return `(classified_df, counts_dict)`.


In [ ]:
def filter_differential_expression(
    filepath: Path,
    lfc_thresh: float = 1.0,
    padj_thresh: float = 0.05,
    save_path: Path = None
) -> tuple[pd.DataFrame, dict[str, int]]:
    # TODO: Classify genes by log2FC and padj thresholds, plot volcano, return (df, counts).
    ...


In [ ]:
# ✅ Test Task C6 — Run me to verify
diff_csv = DATA_DIR / "tables" / "differential_expression.csv"
volcano_png = OUT_DIR / "volcano_plot.png"
de_df, de_counts = filter_differential_expression(diff_csv, save_path=volcano_png)

print("Differential Expression Counts:", de_counts)
assert "regulation" in de_df.columns
assert "Up-regulated" in de_counts
assert "Down-regulated" in de_counts
assert volcano_png.exists()
print("Task C6 passed ✔")


---

## Graded Task C7 (★★☆) · Pathway Mapping & Expression Enrichment Grouping

### Biological Context
Single-gene expression changes are often noisy. To understand biological mechanisms, bioinformaticians map genes to curated **functional pathways** (e.g. KEGG or Reactome) and evaluate collective pathway trends.

### Specification
Implement `pathway_expression_enrichment(expr_filepath: Path) -> pd.DataFrame`:
1. Load `data/tables/expression_matrix.csv`.
2. Map genes to functional biological categories using the following reference dictionary:
   - `TP53`, `ATM`, `BRCA1`, `CHEK2` $\to$ `'DNA Damage & Apoptosis'`
   - `EGFR`, `ERBB2`, `MET` $\to$ `'Receptor Tyrosine Kinase'`
   - `MYC`, `CCND1`, `CDK4` $\to$ `'Cell Cycle & Proliferation'`
   - All other genes $\to$ `'Other Metabolism & Signaling'`
3. Compute `mean_expr` across sample columns for each gene.
4. Group by `'pathway'` and calculate `count`, `mean`, `median`, and `std` of `mean_expr`.
5. Return the summary DataFrame sorted descending by `mean`.


In [ ]:
def pathway_expression_enrichment(expr_filepath: Path) -> pd.DataFrame:
    # TODO: Map genes to pathways, compute mean expression, group by pathway.
    ...


In [ ]:
# ✅ Test Task C7 — Run me to verify
pw_summary = pathway_expression_enrichment(DATA_DIR / "tables" / "expression_matrix.csv")
print("Pathway Summary Table:\n", pw_summary)

assert isinstance(pw_summary, pd.DataFrame)
assert "pathway" in pw_summary.columns
assert len(pw_summary) >= 3
assert "DNA Damage & Apoptosis" in pw_summary["pathway"].values
print("Task C7 passed ✔")


---

## 6. Summary & Cheat Sheet Table

| Domain | Library | Task / Method | Code Snippet |
|---|---|---|---|
| **File I/O** | `pathlib.Path` | Path traversal | `for p in Path("data").rglob("*.fasta"): ...` |
| **File I/O** | `shutil` | Safe copy with timestamps | `shutil.copy2(src, dst)` |
| **File I/O** | `re` + `Path` | Batch regex rename | `p.rename(f"S{int(m.group(1)):03d}.fasta")` |
| **NumPy** | Array Ops | Fast ASCII indexing | `lookup[np.frombuffer(seq.encode(), dtype=np.uint8)]` |
| **NumPy** | Prefix Sums | $O(N)$ Sliding window | `cumsum = np.cumsum(arr); counts = cumsum[w:] - cumsum[:-w]` |
| **NumPy** | Broadcasting | Gene-wise Z-score | `(mat - mat.mean(axis=1, keepdims=True)) / stds_safe` |
| **NumPy** | Motif Matrices | PFM & PWM | `counts / n; np.log2(P / background)` |
| **NumPy** | Strides | k-mer spectrum | `sliding_window_view(seq_bytes, window_shape=k)` |
| **NumPy** | Substitution | Ti / Tv ratio | `ti = ((b1=='A')&(b2=='G')) | ...; ti_count / tv_count` |
| **pandas** | Tabular Filter | Expression thresholding | `df[df[sample_cols].mean(axis=1) > threshold]` |
| **pandas** | Relational Join | Inner merge | `pd.merge(expr_df, annot_df, on="gene", how="inner")` |
| **pandas** | Aggregation | MultiIndex Groupby | `df.groupby(["tissue", "gene"])["tpm"].agg(["mean", "std"])` |
| **pandas** | Reshape | Wide to Long & Pivot | `df.melt(...); df.pivot_table(index="gene", columns="tissue")` |
| **pandas** | Classification | Volcano categorisation | `df.loc[(df["lfc"]>=1) & (df["padj"]<0.05), "reg"] = "Up"` |
| **Seaborn** | Visualization | Publication plots | `sns.boxplot(..., hue="condition"); sns.scatterplot(...)` |

---

## 7. Further Reading & References

1. **Python Pathlib Documentation:** [docs.python.org/3/library/pathlib.html](https://docs.python.org/3/library/pathlib.html)
2. **NumPy Broadcasting & Strides:** [numpy.org/doc/stable/user/basics.broadcasting.html](https://numpy.org/doc/stable/user/basics.broadcasting.html)
3. **pandas Documentation:** [pandas.pydata.org/docs/user_guide/](https://pandas.pydata.org/docs/user_guide/)
4. **Biopython Tutorial & Cookbook:** [biopython.org/DIST/docs/tutorial/Tutorial.html](https://biopython.org/DIST/docs/tutorial/Tutorial.html)
